# DiD-BCF — D_contamination (linearity_degree=2)

**Workstream D · staggered adoption (cohort x event-time effects)**

stronger dynamics -> larger TWFE contamination

Fits DiD-BCF on the `D_contamination` scenario at `linearity_degree=2` and reports
metrics for **both** the plain DiD-BCF posterior and the proposed **posterior
correction** (Algorithm 1 of the theory note), so the correction can be judged
directly. Panel: N=200, 4 pre + 4 post periods.

> **Colab:** upload just this notebook and *Run all* — the first cell installs the
> dependencies and the second clones the engine automatically.

In [1]:
# Colab: install the DiD-BCF dependencies (stochtree provides the BCF sampler).
%pip install -q stochtree scikit-learn joblib tqdm pandas numpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 5.4 MB/s eta 0:00:00


In [2]:
import os, sys

# --- Locate the DiD-BCF engine ------------------------------------------------
# So you can upload just THIS notebook to Colab and Run all. Resolution order:
#   1. `did_bcf_revision` already importable;
#   2. running inside a repo checkout (the parent folder holds the package);
#   3. otherwise clone https://github.com/hugogobato/DiD-BCF and use it.
REPO_URL = "https://github.com/hugogobato/DiD-BCF.git"
ENGINE_SUBDIR = os.path.join("DiD-BCF", "Simulation_Studies_Revision")

def _locate_root():
    try:
        import did_bcf_revision  # noqa: F401
        return os.path.dirname(os.path.dirname(did_bcf_revision.__file__))
    except Exception:
        pass
    parent = os.path.abspath(os.path.join(os.getcwd(), ".."))
    if os.path.isdir(os.path.join(parent, "did_bcf_revision")):
        return parent
    if not os.path.isdir("DiD-BCF"):
        import subprocess
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL], check=True)
    return os.path.abspath(ENGINE_SUBDIR)

ROOT = _locate_root()
sys.path.insert(0, ROOT)
print("Using DiD-BCF engine at:", ROOT)

from did_bcf_revision.runner import run_named
from did_bcf_revision.metrics import (compute_metrics, plain_vs_corrected,
                                      surface_metrics)

Using DiD-BCF engine at: /content/DiD-BCF/Simulation_Studies_Revision


In [3]:
REPS = 100      # replications (lower for a quick smoke test)
JOBS = 1        # parallel reps (keep 1 on a single-core/GPU Colab)

bcf_params = dict(num_gfr=50, num_mcmc=500, keep_every=5, num_chains=3)

summaries = run_named(
    "D_contamination",
    linearity_degree=2,
    reps=REPS,
    jobs=JOBS,
    bcf_params=bcf_params,
    prop_method="logit",   # pilot propensity for the posterior correction
    n_splits=2,            # cross-fitting folds for the correction
)
summaries.head()

[D_contamination_lin_2] staggered DGP | N=(200,) | reps=100 | 100 fits | jobs=1


D_contamination: 100%|██████████| 100/100 [4:17:25<00:00, 154.46s/fit]

[D_contamination_lin_2] wrote 3000 rows -> /content/DiD-BCF/Simulation_Studies_Revision/Results/summaries_D_contamination_lin_2.csv


,dgp,setting,linearity_degree,N,rep,estimand_type,estimand_id,g,t,k,...,p_bayes,surf_rmse,surf_mae,surf_n,surf_mape,surf_cover95,surf_len95,surf_cover90,surf_len90,true
0,staggered,D_contamination,2,200,0,GATT,g=4_t=4,4.0,4.0,0.0,...,0.273333,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.732024
1,staggered,D_contamination,2,200,0,GATT,g=4_t=5,4.0,5.0,1.0,...,0.075333,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4.917643
2,staggered,D_contamination,2,200,0,GATT,g=4_t=6,4.0,6.0,2.0,...,0.020000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,7.103263
3,staggered,D_contamination,2,200,0,GATT,g=4_t=7,4.0,7.0,3.0,...,0.000667,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,9.288882
4,staggered,D_contamination,2,200,0,GATT,g=5_t=5,5.0,5.0,0.0,...,0.026667,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5.529720


In [4]:
# Decomposed metrics: bias, MC SD/variance, RMSE, MAE, MAPE, coverage 90/95,
# interval length, calibration ratio (avg_post_sd/emp_sd), size/power and their
# Monte-Carlo SEs -- for plain AND corrected DiD-BCF.
metrics = compute_metrics(summaries)
plain_vs_corrected(metrics)

,dgp,setting,linearity_degree,N,estimand_type,estimand_id,role,bias_corrected,bias_plain,cover90_corrected,...,len95_corrected,len95_plain,mae_corrected,mae_plain,reject05_corrected,reject05_plain,rmse_corrected,rmse_plain,sd_ratio_corrected,sd_ratio_plain
0,staggered,D_contamination,2,200,ATT,ATT,power,-2.942689,-6.482195,0.00,...,0.879548,2.667885,2.942689,6.482195,1.0,0.95,2.989050,6.499309,0.386452,1.525726
1,staggered,D_contamination,2,200,ES,k=0,power,-1.053739,-4.702451,0.03,...,0.827814,1.376958,1.055260,4.702451,1.0,0.45,1.121421,4.711758,0.500420,1.783750
2,staggered,D_contamination,2,200,ES,k=1,power,-3.895683,-7.718728,0.00,...,0.996598,2.709936,3.895683,7.718728,1.0,0.96,3.942197,7.740245,0.412237,1.514851
3,staggered,D_contamination,2,200,ES,k=2,power,-4.234909,-7.605794,0.00,...,1.252834,3.889128,4.234909,7.605794,1.0,0.95,4.289400,7.635249,0.406517,1.457336
4,staggered,D_contamination,2,200,ES,k=3,power,-3.074501,-5.834307,0.00,...,1.672610,4.839667,3.074501,5.834307,1.0,0.96,3.171095,5.870930,0.467699,1.697280
5,staggered,D_contamination,2,200,GATT,g=4_t=4,power,-0.024641,-2.747527,0.85,...,1.428000,2.175152,0.332570,2.747527,1.0,0.00,0.410591,2.750138,0.838178,29.232756
6,staggered,D_contamination,2,200,GATT,g=4_t=5,power,-1.204798,-4.013976,0.13,...,1.460771,2.997500,1.210021,4.013976,1.0,0.08,1.326052,4.037470,0.603635,1.679424
7,staggered,D_contamination,2,200,GATT,g=4_t=6,power,-2.271105,-5.047574,0.02,...,1.536093,4.069869,2.271105,5.047574,1.0,0.51,2.377132,5.081482,0.501068,1.688298
8,staggered,D_contamination,2,200,GATT,g=4_t=7,power,-3.074501,-5.834307,0.00,...,1.672610,4.839667,3.074501,5.834307,1.0,0.96,3.171095,5.870930,0.467699,1.697280
9,staggered,D_contamination,2,200,GATT,g=5_t=5,power,-0.659089,-4.571467,0.53,...,1.684489,2.131404,0.677431,4.571467,1.0,0.37,0.811220,4.595658,0.823297,1.283780


## CATT-surface metrics (the paper's headline RMSE/MAE/MAPE)

Within-replication RMSE/MAE/MAPE over the *individual* treated observations
(mean +/- SD across runs) plus the *pointwise* CATT coverage -- the evidence
that DiD-BCF recovers the heterogeneous effect that GATT-only methods cannot.

In [5]:
surface_metrics(summaries)

,dgp,setting,linearity_degree,N,method,n_reps,avg_n_treated_obs,surf_rmse_mean,surf_rmse_sd,surf_mae_mean,...,surf_mape_mean,surf_mape_sd,surf_cover90_mean,surf_cover90_sd,surf_cover95_mean,surf_cover95_sd,surf_len90_mean,surf_len90_sd,surf_len95_mean,surf_len95_sd
0,staggered,D_contamination,2,200,corrected,100,431.6,4.784047,0.415086,3.423388,...,0.356551,0.026661,0.178812,0.025917,0.215572,0.030888,1.417789,0.125281,1.714615,0.161367
1,staggered,D_contamination,2,200,plain,100,431.6,7.526756,0.504431,6.482195,...,0.781391,0.053686,0.012255,0.016578,0.024964,0.022552,3.224436,0.291372,3.855044,0.330184


## Goodman-Bacon decomposition (TWFE contamination)

How much of a naive TWFE estimate on this DGP comes from the
"already-treated as control" comparisons that bias it.

In [6]:
from did_bcf_revision.dgps import generate_staggered_did
from did_bcf_revision.goodman_bacon import bacon_summary

df0 = generate_staggered_did(seed=0, linearity_degree=2)
bacon_summary(df0)

{'twfe': 5.063198924628939,
 'bacon_total': 5.06319892462894,
 'weight_treated_vs_untreated': 0.6334173693819004,
 'weight_earlier_vs_later': 0.2391665942217972,
 'weight_already_treated': 0.1274160363963024,
 'components':                    type  treat_group  control_group    weight        dd  \
 0      Earlier_vs_Later          4.0            5.0  0.060205  2.641511   
 1      Earlier_vs_Later          4.0            6.0  0.106778  3.036811   
 2      Earlier_vs_Later          5.0            6.0  0.072184  4.176324   
 3      Later_vs_Earlier          5.0            4.0  0.045153  3.585888   
 4      Later_vs_Earlier          6.0            4.0  0.053389  4.402213   
 5      Later_vs_Earlier          6.0            5.0  0.028874  4.134568   
 6  Treated_vs_Untreated          4.0            inf  0.231731  4.719428   
 7  Treated_vs_Untreated          5.0            inf  0.234982  6.264973   
 8  Treated_vs_Untreated          6.0            inf  0.166704  7.176302   
 
    contributio